# SIXray-D: student and feature KD

Compare the same ResNet-18 initialization with and without a frozen RT-DETRv2 teacher

In [3]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'students/anna_student_resnet18/src')]

from src.utils.checkpoint import load_checkpoint
from src.engine.inference import prepare_validation_files
from sixray_student.data import SixRayStudentDataset, detection_collate_fn, load_split
from sixray_student.model import build_student_model
from sixray_student.config import CLASS_NAMES, BBOX_ENCODING
from sixray_student.metrics import evaluate_map
from sixray_student.train_utils import train_one_epoch, make_grad_scaler

c:\Users\89526\Documents\GitHub\sixray-kd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and training settings

In [5]:
ARTIFACT_ROOT = Path('C:/Users/89526/Documents/sixray-artifacts')
PROJECT_DIR = ARTIFACT_ROOT / 'DatasetAPAI/SIXray_Project'
DATA_DIR = ARTIFACT_ROOT / 'DatasetAPAI/subset_clean'
SPLIT_PATH = PROJECT_DIR / 'splits/split_seed42_train10500_val300pos1200neg.json'
TEACHER_PATH = PROJECT_DIR / 'checkpoints_angelo/02_rtdetr_teacher_oversampling_angelo_best'
INITIAL_PATH = ARTIFACT_ROOT / 'experiments/feature_kd_20261016T214646677055Z/initial_student.pth'
RESULTS_DIR = ARTIFACT_ROOT / 'experiments/matched_20261016T221317947683Z'

SEED, ADAPTER_SEED = 42, 43
EPOCHS, BATCH_SIZE, NUM_WORKERS = 50, 2, 2
LEARNING_RATE, WEIGHT_DECAY, MIN_LR = 1e-4, 1e-4, 1e-6
KD_WEIGHT, BACKGROUND_WEIGHT = 1.0, 0.1
SCORE_THRESHOLD, NMS_IOU, MAX_DETECTIONS = 0.001, 0.5, 100
RUN_TRAINING = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Validation split and DataLoaders

In [ ]:
train_indices, val_indices, _ = load_split(SPLIT_PATH)

if RUN_TRAINING:
    RESULTS_DIR.mkdir(parents=True, exist_ok=False)
    prepare_validation_files(ARTIFACT_ROOT / 'DatasetAPAI/subset_clean.zip', DATA_DIR,
                             train_indices + val_indices)
    
train_data = SixRayStudentDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', train_indices)
val_data = SixRayStudentDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', val_indices)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(SEED), collate_fn=detection_collate_fn)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        generator=torch.Generator().manual_seed(SEED), collate_fn=detection_collate_fn)

## Frozen teacher and feature loss

The teacher receives the resized `[0,1]` image directly; the student's ImageNet normalization stays inside its backbone path. KD compares teacher `model.backbone.model(...).feature_maps[-1]` `[B,2048,20,20]` with student `backbone(normalizer(...))` `[B,512,20,20]`. The teacher's encoder and decoder are not run for KD.

A training-only 1?1 convolution maps 512 to 2048 channels. `feature_distillation_loss` L2-normalizes channels at each cell, sums squared channel differences, then averages foreground and background separately: `L_KD = mean(L_fg + 0.1 L_bg)`. The adapter is not needed for inference.

An original box corner `(x,y)` becomes `(640x/W,640y/H)` in the resized image and `(20x/W,20y/H)` on the feature grid. Floor left/top and ceil right/bottom; take the union of all boxes. Negative images contribute only background KD. The unchanged detection objective is `L_obj + 10 L_box + L_class`; KD adds `1.0 L_KD`.

The saved oversampling teacher is used as-is; its epoch-28 label does not establish a valid continuation from baseline training because the historical resume code did not restore model weights.

In [ ]:
_, teacher, _ = load_checkpoint(TEACHER_PATH, device=DEVICE, strict=True)
teacher.requires_grad_(False).eval();

## Shared training loop

This function is used twice so initialization, optimizer, data order and validation stay matched. 

Validation is in original-image coordinates after every epoch. Save the best student by COCO mAP@[0.50:0.95]

In [ ]:
def train_student(name, use_kd):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    student = build_student_model(pretrained_backbone=False).to(DEVICE)
    student.load_state_dict(torch.load(INITIAL_PATH, map_location=DEVICE, weights_only=True))
    adapter = None
    parameters = list(student.parameters())
    if use_kd:
        with torch.random.fork_rng():
            torch.manual_seed(ADAPTER_SEED)
            adapter = torch.nn.Conv2d(512, 2048, 1, bias=False).to(DEVICE)
        parameters += list(adapter.parameters())
    train_loader.generator.manual_seed(SEED)
    val_loader.generator.manual_seed(SEED)
    optimizer = torch.optim.AdamW(parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    scaler = make_grad_scaler(DEVICE)
    run_dir = RESULTS_DIR / name
    run_dir.mkdir()
    history, best_map = [], -1.0
    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(
            student, train_loader, optimizer, DEVICE, epoch, scaler,
            use_amp=True, grad_clip_norm=1.0, log_every_n_batches=0, scheduler=scheduler,
            teacher=teacher if use_kd else None, adapter=adapter,
            kd_weight=KD_WEIGHT, background_weight=BACKGROUND_WEIGHT,
        )
        validation = evaluate_map(
            student, val_loader, DEVICE, confidence_threshold=SCORE_THRESHOLD,
            nms_iou_threshold=NMS_IOU, max_detections=MAX_DETECTIONS,
            use_amp=False, original_coordinates=True,
        )
        history.append({'epoch': epoch, 'train': train_metrics, 'validation': validation})
        if validation['map'] > best_map:
            best_map = validation['map']
            torch.save({
                'epoch': epoch, 'model_state_dict': student.state_dict(),
                'adapter_state_dict': adapter.state_dict() if use_kd else None,
                'validation': validation, 'class_names': CLASS_NAMES,
                'image_size': 640, 'grid_size': 20, 'num_classes': 5, 'num_boxes': 2,
                'bbox_encoding': BBOX_ENCODING, 'best_checkpoint_metric': 'map',
            }, run_dir / 'student_best_map.pth')
        with (run_dir / 'history.json').open('w') as output:
            json.dump(history, output, indent=2)
        print(f"{name} {epoch}/{EPOCHS}: detection={train_metrics['detection_loss']:.4f}, "
              f"KD={train_metrics['kd_loss']:.4f}, mAP={validation['map']:.4f}")

## Supervised control

Record the shared settings once, then train without a teacher loss. This cell and the KD cell remain disabled unless `RUN_TRAINING` is enabled.

In [ ]:
if RUN_TRAINING:
    with (RESULTS_DIR / 'settings.json').open('x') as output:
        json.dump({
            'seed': SEED, 'adapter_seed': ADAPTER_SEED, 'split_path': str(SPLIT_PATH),
            'data_dir': str(DATA_DIR), 'teacher_path': str(TEACHER_PATH), 'initial_path': str(INITIAL_PATH),
            'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS,
            'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
            'scheduler': 'cosine per 5250 successful updates', 'min_lr': MIN_LR, 'grad_clip_norm': 1.0,
            'kd_weight': KD_WEIGHT, 'background_weight': BACKGROUND_WEIGHT,
            'preprocessing': 'resize_only_640; student internal ImageNet normalization',
            'training_amp': DEVICE.type == 'cuda', 'evaluation_amp': False,
            'score_threshold': SCORE_THRESHOLD, 'nms_iou': NMS_IOU, 'max_detections': MAX_DETECTIONS,
            'evaluation_frame': 'original_image_xyxy', 'selection_metric': 'map',
            'control_checkpoint': 'control/student_best_map.pth', 'kd_checkpoint': 'kd/student_best_map.pth',
        }, output, indent=2)
    train_student('control', use_kd=False)

## Feature KD

Reload the same initial student with the frozen teacher, the adapter and the KD term.

In [ ]:
if RUN_TRAINING:
    train_student('kd', use_kd=True)

## Compare selected checkpoints

Read the full-validation metrics stored when each best checkpoint was selected. This also displays the completed matched experiment without rerunning inference or evaluation. AP50, AP75 and per-class AP/AR remain in each checkpoint and history file.

In [6]:
print(f"{'Model':<12} {'Epoch':>6} {'mAP':>8} {'AP50':>8} {'AP75':>8}")
for name in ('control', 'kd'):
    checkpoint = torch.load(RESULTS_DIR / name / 'student_best_map.pth', map_location='cpu', weights_only=False)
    metrics = checkpoint['validation']
    print(f"{name:<12} {checkpoint['epoch']:>6} {metrics['map']:>8.4f} "
          f"{metrics['map_50']:>8.4f} {metrics['map_75']:>8.4f}")

Model         Epoch      mAP     AP50     AP75
control          46   0.3813   0.6655   0.3859
kd               45   0.4043   0.6882   0.4075
